# Geodemographic Clustering - Median Run (500 Epochs, Linear Scaling)

This notebook is identical to notebook 6a (Geodemographic Clustering - Median Run) but uses the
**500-epoch linear scaling** retraining stability runs (`retraining_stability_500epochs_linscaling`)
instead of the default 250-epoch runs.

**Key difference from notebook 6a:**
- AE embeddings are loaded from the 500-epoch stability checkpoint (median run)
- All outputs are saved to a separate directory to avoid overwriting other results

**Purpose:**
- Verify that clustering results hold with the improved 500-epoch training
- The median run is chosen as it represents typical model behavior

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import clustergram
import joblib
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')


## Configuration

In [ ]:
# Parameters
target_dim = 64
n_clusters = 8  # Match OAC supergroups
random_seed = 20210321

# Paths
cleaned_data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
oac_path = "../data/OAC/OAC_assignment.csv"

# Stability checkpoint path (500 epochs, linear scaling)
stability_path = f"../AE_outputs/retraining_stability_500epochs_linscaling/data/stability_checkpoint_{target_dim}d.pkl"

# Output directories
plots_dir = "./plots/clustering_plots/"
results_dir = "./clustering_results/"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print(f"Creating {n_clusters} clusters from {target_dim}D embeddings (500 EPOCH MEDIAN RUN)")
print(f"Random seed: {random_seed}")
print(f"Plots directory: {plots_dir}")
print(f"Results directory: {results_dir}")

## Load Data

In [3]:
# Load original census data
print("Loading census data...")
df_census = pd.read_parquet(cleaned_data_path)
df_census = df_census.set_index('OA')
print(f"Census data: {df_census.shape}")

# Load PCA embeddings (compute on the fly)
print(f"\nComputing PCA {target_dim}D embeddings...")
pca = PCA(n_components=target_dim, random_state=random_seed)
pca_embeddings = pca.fit_transform(df_census)
pca_latent_df = pd.DataFrame(pca_embeddings, index=df_census.index)
print(f"PCA embeddings: {pca_latent_df.shape}")
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# Load AE embeddings from 500-epoch stability checkpoint (MEDIAN RUN)
print(f"\nLoading stability checkpoint from: {stability_path}")
with open(stability_path, 'rb') as f:
    checkpoint = pickle.load(f)

embeddings_list = checkpoint['embeddings_list']
reco_errors_list = checkpoint['reco_errors_list']
n_runs = checkpoint['n_runs']
print(f"Loaded {n_runs} runs from stability checkpoint")

# Find the median run (by overall RMSE)
rmse_per_run = []
for run_idx, reco_err in enumerate(reco_errors_list):
    rmse = np.sqrt(reco_err.mean()) * 100
    rmse_per_run.append(rmse)

rmse_per_run = np.array(rmse_per_run)
median_rmse = np.median(rmse_per_run)
median_run_idx = np.argmin(np.abs(rmse_per_run - median_rmse))

print(f"\nRMSE per run: {rmse_per_run}")
print(f"Median RMSE: {median_rmse:.4f}%")
print(f"Median run index: {median_run_idx} (RMSE = {rmse_per_run[median_run_idx]:.4f}%)")

# Get median run embeddings
ae_embeddings = embeddings_list[median_run_idx]
ae_latent_df = pd.DataFrame(ae_embeddings, index=df_census.index)
print(f"\nAE embeddings (median run): {ae_latent_df.shape}")

# Load OAC
print(f"\nLoading OAC classifications...")
oac = pd.read_csv(oac_path)
oac = oac[['Geography_Code', 'Supergroup8']].rename(columns={'Geography_Code': 'OA'})
oac = oac.set_index('OA')
oac = oac.reindex(df_census.index)
print(f"OAC: {oac.shape}")
print(f"OAC Supergroups: {oac['Supergroup8'].nunique()} unique groups")


Loading census data...
Census data: (188880, 408)

Computing PCA 64D embeddings...
PCA embeddings: (188880, 64)
Explained variance: 0.968

Loading stability checkpoint from: ../AE_outputs/retraining_stability_500epochs_linscaling/data/stability_checkpoint_64d.pkl
Loaded 10 runs from stability checkpoint

RMSE per run: [1.10684862 1.09978463 1.10845555 1.10463604 1.10444621 1.10335998
 1.10721062 1.10775274 1.11074648 1.10391072]
Median RMSE: 1.1057%
Median run index: 3 (RMSE = 1.1046%)

AE embeddings (median run): (188880, 64)

Loading OAC classifications...
OAC: (188880, 1)
OAC Supergroups: 8 unique groups


In [ ]:
# Make a clustergram for PCA and AE embeddings
print("\nGenerating PCA clustergram...")
cgram_pca = clustergram.Clustergram(range(1,10), n_init=10, random_state=random_seed)
cgram_pca.fit(pca_latent_df)

# Create clustergram for AE embeddings  
print("Generating AE clustergram...")
cgram_ae = clustergram.Clustergram(range(1,10), n_init=10, random_state=random_seed)
cgram_ae.fit(ae_latent_df)

# Plot both clustergrams side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# PCA clustergram
cgram_pca.plot(ax=axes[0], figsize=None)
axes[0].set_title(f'PCA {target_dim}D Embeddings Clustergram', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of clusters (k)', fontsize=12)
axes[0].set_ylabel('Cluster', fontsize=12)
axes[0].axvline(x=n_clusters, color='red', linestyle='--', linewidth=2, label=f'k={n_clusters} (target)')
axes[0].legend()

# AE clustergram
cgram_ae.plot(ax=axes[1], figsize=None)
axes[1].set_title(f'AE {target_dim}D Embeddings Clustergram', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of clusters (k)', fontsize=12)
axes[1].set_ylabel('Cluster', fontsize=12)
axes[1].axvline(x=n_clusters, color='red', linestyle='--', linewidth=2, label=f'k={n_clusters} (target)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{plots_dir}/clustergram_comparison_{target_dim}d.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Clustergrams saved to {plots_dir}/clustergram_comparison_{target_dim}d.png")

In [ ]:
import time
from datetime import datetime, timedelta

n_inits = 10000
clustering_file = f"{results_dir}/cluster_assignments_{target_dim}d_k{n_clusters}_n{n_inits}.csv"
pca_model_file = f"{results_dir}/kmeans_pca_{target_dim}d_k{n_clusters}_n{n_inits}.joblib"
ae_model_file = f"{results_dir}/kmeans_ae_{target_dim}d_k{n_clusters}_n{n_inits}.joblib"


# --- PCA Clustering (check separately) ---
if os.path.exists(pca_model_file):
    print(f"Loading existing PCA clustering from {pca_model_file}...")
    kmeans_pca = joblib.load(pca_model_file)
    pca_clusters = kmeans_pca.predict(pca_latent_df)
    print(f"  KMeans PCA - Inertia: {kmeans_pca.inertia_:.2f}")
else:
    estimated_finish = datetime.now() + timedelta(seconds=n_inits)
    print(f"Performing PCA K-Means clustering (k={n_clusters}, n_init={n_inits})...")
    print(f"  Estimated finish: {estimated_finish.strftime('%H:%M:%S')}")
    start_time = time.time()
    kmeans_pca = KMeans(n_clusters=n_clusters, random_state=random_seed, n_init=n_inits, max_iter=1000)
    pca_clusters = kmeans_pca.fit_predict(pca_latent_df)
    pca_time = time.time() - start_time
    print(f"  Time: {pca_time:.1f}s")
    print(f"  Inertia: {kmeans_pca.inertia_:.2f}")
    print(f"  Iterations: {kmeans_pca.n_iter_}")
    joblib.dump(kmeans_pca, pca_model_file)
    print(f"  Saved to {pca_model_file}")

    
# --- AE Clustering (check separately) ---
if os.path.exists(ae_model_file):
    print(f"\nLoading existing AE clustering from {ae_model_file}...")
    kmeans_ae = joblib.load(ae_model_file)
    ae_clusters = kmeans_ae.predict(ae_latent_df)
    print(f"  KMeans AE - Inertia: {kmeans_ae.inertia_:.2f}")
else:
    estimated_finish = datetime.now() + timedelta(seconds=n_inits)
    print(f"\nPerforming AE K-Means clustering (k={n_clusters}, n_init={n_inits})...")
    print(f"  Estimated finish: {estimated_finish.strftime('%H:%M:%S')}")
    start_time = time.time()
    kmeans_ae = KMeans(n_clusters=n_clusters, random_state=random_seed, n_init=n_inits, max_iter=1000)
    ae_clusters = kmeans_ae.fit_predict(ae_latent_df)
    ae_time = time.time() - start_time
    print(f"  Time: {ae_time:.1f}s")
    print(f"  Inertia: {kmeans_ae.inertia_:.2f}")
    print(f"  Iterations: {kmeans_ae.n_iter_}")
    joblib.dump(kmeans_ae, ae_model_file)
    print(f"  Saved to {ae_model_file}")



# --- Create/Update clusters_df ---
clusters_df = pd.DataFrame({
    'pca_cluster': pca_clusters,
    'ae_cluster': ae_clusters,
    'oac_supergroup': oac['Supergroup8'].values
}, index=df_census.index)

# Save combined results
clusters_df.to_csv(clustering_file)
print(f"\nCluster assignments saved to {clustering_file}")
print("\nCluster distributions:")
print("\nPCA clusters:")
print(clusters_df['pca_cluster'].value_counts().sort_index())
print("\nAE clusters:")
print(clusters_df['ae_cluster'].value_counts().sort_index())